Bibliotecas

In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms
from scipy import stats

In [5]:
import faraway.datasets.galapagos
galapagos = faraway.datasets.galapagos.load()

In [ ]:
galapagos = pd.read_csv("dados/galapagos.csv")
galapagos.head()

In [17]:
model = smf.ols("Species ~ Area + Elevation + Nearest + Scruz + Adjacent", data=galapagos).fit()

In [18]:
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                Species   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.717
Method:                 Least Squares   F-statistic:                     15.70
Date:                Thu, 18 Jun 2026   Prob (F-statistic):           6.84e-07
Time:                        21:55:14   Log-Likelihood:                -162.54
No. Observations:                  30   AIC:                             337.1
Df Residuals:                      24   BIC:                             345.5
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      7.0682     19.154      0.369      0.715     -32.464      46.601
Area          -0.0239      0.022     -1.068      0.296      -0.070       0.022
Elevation      0.3195      0.054      5.953      0.000       0.209       0.430
Nearest        0.0091      1.054      0.009      0.993      -2.166       2.185
Scruz         -0.2405      0.215     -1.117      0.275      -0.685       0.204
Adjacent      -0.0748      0.018     -4.226      0.000      -0.111      -0.038
==============================================================================
Omnibus:                       12.683   Durbin-Watson:                   2.476
Prob(Omnibus):                  0.002   Jarque-Bera (JB):               13.498
Skew:                           1.136   Prob(JB):                      0.00117
Kurtosis:                       5.374   Cond. No.                     1.90e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.9e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

$$RSS = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

In [22]:
sms.anova_lm(model)

,df,sum_sq,mean_sq,F,PR(>F)
Area,1.0,145470.212144,145470.212144,39.126209,0.000002
Elevation,1.0,65664.304683,65664.304683,17.661315,0.000315
Nearest,1.0,29.242630,29.242630,0.007865,0.930067
Scruz,1.0,14279.853308,14279.853308,3.840762,0.061732
Adjacent,1.0,66406.387572,66406.387572,17.860909,0.000297
Residual,24.0,89231.366330,3717.973597,NaN,NaN


## Álgebra Matricial

### Montagem de $X$, $y$ e da matriz chapéu $H$

$$H = X(X^\top X)^{-1}X^\top$$

In [39]:
n = len(galapagos)           # numero de observações
k = 5                        # numero de preditores (sem contar o intercepto)

X = np.column_stack([
    np.ones(n), # coluna de 1s
    galapagos["Area"],
    galapagos["Elevation"],
    galapagos["Nearest"],
    galapagos["Scruz"],
    galapagos["Adjacent"]
])

y = galapagos["Species"].values

H = X@np.linalg.inv(X.T @ X)@X.T

print(X[:5])

[[1.0000e+00 2.5090e+01 3.4600e+02 6.0000e-01 6.0000e-01 1.8400e+00]
 [1.0000e+00 1.2400e+00 1.0900e+02 6.0000e-01 2.6300e+01 5.7233e+02]
 [1.0000e+00 2.1000e-01 1.1400e+02 2.8000e+00 5.8700e+01 7.8000e-01]
 [1.0000e+00 1.0000e-01 4.6000e+01 1.9000e+00 4.7400e+01 1.8000e-01]
 [1.0000e+00 5.0000e-02 7.7000e+01 1.9000e+00 1.9000e+00 9.0382e+02]]


In [33]:
X.shape # dimensão X

(30, 6)

In [34]:
H.shape # dimensão H

(30, 30)

### Soma de Quadrados dos Resíduos (SQRes) e estimativa de $\sigma^2$

$$SQRes = y^\top(I - H)y$$

$$\hat{\sigma}^2 = \frac{SQRes}{n - k - 1}$$

In [38]:
In = np.eye(n)

SQRes = y@(In - H)@y
print(f"SQRes = {SQRes:.4f}")      

sigma2h = SQRes / (n - k - 1)
print(f"\nsigma² = {sigma2h:.4f}")
print(f"sigma = {np.sqrt(sigma2h):.4f}")

SQRes = 89231.3663

sigma²  = 3717.9736
sigma   = 60.9752


### Soma de Quadrados da Regressão (SQReg) e Teste F Global

$$SQReg = y^\top\left(H - \frac{J}{n}\right)y \quad \text{onde } J = \mathbf{1}\mathbf{1}^\top$$

A matriz $J/n$ centraliza $y$ em torno de $\bar{y}$, de modo que $SQReg$ mede o quanto a regressão explica além da média.

A estatística $F$ global testa $H_0: \beta_1 = \cdots = \beta_k = 0$:

$$F = \frac{SQReg / k}{SQRes / (n - k - 1)}$$

In [42]:
J = np.ones((n, n))

SQReg = y@(H - J/n)@y
print(f"SQReg = {SQReg:.4f}")

F_global = (SQReg/k)/(SQRes/(n - k - 1))
p_valor = 1 - stats.f.cdf(F_global, dfn=k, dfd=n - k - 1)

print(f"\nF = {F_global:.4f}")
print(f"p-valor = {p_valor:.6f}")

SQReg = 291850.0003

F = 15.6994
p-valor = 0.000001


### Estimativa dos coeficientes $\hat{\beta}$ e seus erros padrão

$$\hat{\beta} = (X^\top X)^{-1} X^\top y$$

A matriz de covariâncias de $\hat{\beta}$:

$$\text{Var}(\hat{\beta}) = (X^\top X)^{-1} \hat{\sigma}^2$$

O erro padrão de cada $\hat{\beta}_j$ é a raiz quadrada do $j$-ésimo elemento da diagonal.

In [59]:
XtX_inv = np.linalg.inv(X.T @ X)
betah = XtX_inv @ X.T @ y

varbeta = XtX_inv * sigma2h

sdbeta = np.sqrt(np.diag(varbeta))

nomes = ["Intercepto", "Area", "Elevation", "Nearest", "Scruz", "Adjacent"]
print(f"{'Coef':<12} {'beta_hat':>12} {'std_err':>12}")
print("-" * 38)
for nome, b, s in zip(nomes, betah, sdbeta):
    print(f"{nome:<12} {b:>12.4f} {s:>12.4f}")

Coef             beta_hat      std_err
--------------------------------------
Intercepto         7.0682      19.1542
Area              -0.0239       0.0224
Elevation          0.3195       0.0537
Nearest            0.0091       1.0541
Scruz             -0.2405       0.2154
Adjacent          -0.0748       0.0177


### Estatística $t$ para o intercepto

$$t_{\beta_0} = \frac{\hat{\beta}_0}{\widehat{\text{dp}}(\hat{\beta}_0)}$$

In [62]:
tbeta0 = betah[0] / sdbeta[0]
print(f"t do intercepto = {tbeta0:.4f}")

tbeta0_check = model.params["Intercept"] / sdbeta[0]
print(f"Confirmação = {tbeta0_check:.4f}")

t do intercepto = 0.3690
Confirmação = 0.3690


---
## ANOVA Sequencial (Tipo I)

A ANOVA sequencial decompõe $SQReg$ variável por variável.
A contribuição de cada variável é obtida pela **diferença** entre $SQReg$ dos modelos aninhados:

$$SQ(\beta_j \mid \beta_0, \ldots, \beta_{j-1}) = SQReg_{j} - SQReg_{j-1}$$

### Modelo com só Area

In [63]:
x1 = np.column_stack([np.ones(n), galapagos["Area"]])
beta1h = np.linalg.inv(x1.T @ x1) @ x1.T @ y
H1 = x1@np.linalg.inv(x1.T@x1)@x1.T
SQReg1 = y@(H1 - J / n)@y

print(f"SQReg(Area) = {SQReg1:.4f}")

SQReg(Area) = 145470.2121


### Modelo com Area + Elevation

In [66]:
x2 = np.column_stack([np.ones(n),
                      galapagos["Area"],
                      galapagos["Elevation"]])
beta2h = np.linalg.inv(x2.T@x2)@x2.T@y
H2 = x2@np.linalg.inv(x2.T@x2)@x2.T
SQReg2 = y@(H2 - J/n)@y

print(f"SQReg(Area + Elevation) = {SQReg2:.4f}")
print(f"SQ(Elevation|Area) = SQReg2 - SQReg1 = {SQReg2 - SQReg1:.4f}")

SQReg(Area + Elevation) = 211134.5168
SQ(Elevation|Area) = SQReg2 - SQReg1 = 65664.3047


### Modelo com Area + Elevation + Nearest

In [67]:
x3 = np.column_stack([np.ones(n),
                      galapagos["Area"],
                      galapagos["Elevation"],
                      galapagos["Nearest"]])
beta3h = np.linalg.inv(x3.T @ x3) @ x3.T @ y
H3 = x3 @ np.linalg.inv(x3.T @ x3) @ x3.T
SQReg3 = y @ (H3 - J / n) @ y

print(f"SQReg(Area + Elevation + Nearest)             = {SQReg3:.4f}")
print(f"SQ(Nearest|Area, Elevation) = SQReg3 - SQReg2 = {SQReg3 - SQReg2:.4f}")

SQReg(Area + Elevation + Nearest)             = 211163.7595
SQ(Nearest|Area, Elevation) = SQReg3 - SQReg2 = 29.2426


---
## Teste de Hipótese Geral: $H_0: C\beta = m$

O teste F geral usa a **matriz de contrastes** $C$ para testar hipóteses lineares sobre $\beta$.

$$F = \frac{(C\hat{\beta} - m)^\top \left[C(X^\top X)^{-1}C^\top\right]^{-1}(C\hat{\beta} - m) \;/\; q}{SQRes/(n-k-1)}$$

onde $q$ é o número de restrições (linhas de $C$).

### Teste 1: $H_0: \beta_1 = \beta_2 = \beta_3 = \beta_4 = \beta_5 = 0$

In [68]:
q = 5
C = np.array([
    [0, 1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 1]
], dtype=float)

m = np.zeros(q) # H0: cada beta = 0

diff = C@betah - m
mid  = np.linalg.inv(C@XtX_inv@C.T)

num = diff@mid@diff
F_hip1 = (num/q)/(SQRes/(n - k - 1))
p_hip1 = 1 - stats.f.cdf(F_hip1, dfn=q, dfd=n - k - 1)

print(f"F = {F_hip1:.4f}")
print(f"p-valor = {p_hip1:.6f}")

F = 15.6994
p-valor = 0.000001


### Teste 2: $H_0: \beta_1 = \beta_2 = \beta_3 = \beta_4 = \beta_5$

$$\beta_1 - \beta_2 = 0, \quad \beta_2 - \beta_3 = 0, \quad \beta_3 - \beta_4 = 0, \quad \beta_4 - \beta_5 = 0$$

In [74]:
q2 = 4
C2 = np.array([
    [0,  1, -1,  0,  0,  0],
    [0,  0,  1, -1,  0,  0],
    [0,  0,  0,  1, -1,  0],
    [0,  0,  0,  0,  1, -1]
], dtype=float)

m2 = np.zeros(q2)

diff2 = C2@betah - m2
mid2  = np.linalg.inv(C2@XtX_inv@C2.T)

num2   = diff2@mid2@diff2
F_hip2 = (num2/q2)/(SQRes/(n - k - 1))
p_hip2  = 1 - stats.f.cdf(F_hip2, dfn=q2, dfd=n - k - 1)

print(f"F = {F_hip2:.4f}")
print(f"p-valor = {p_hip2:.6f}")

F = 13.0927
p-valor = 0.000009


---
## Modelos alternativos

### Modelo sem intercepto

In [76]:
# '- 1' remove o intercepto, equivalente ao R: lm(Species ~ ... - 1, galapagos)
model2 = smf.ols("Species ~ Area + Elevation + Nearest + Scruz + Adjacent - 1", data=galapagos).fit()
model2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:                Species   R-squared (uncentered):                   0.850
Model:                            OLS   Adj. R-squared (uncentered):              0.820
Method:                 Least Squares   F-statistic:                              28.38
Date:                Thu, 18 Jun 2026   Prob (F-statistic):                    1.52e-09
Time:                        22:59:18   Log-Likelihood:                         -162.62
No. Observations:                  30   AIC:                                      335.2
Df Residuals:                      25   BIC:                                      342.2
Df Model:                           5                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Area          -0.0266      0.021     -1.280      0.212      -0.070       0.016
Elevation      0.3307      0.044      7.600      0.000       0.241       0.420
Nearest        0.0259      1.035      0.025      0.980      -2.105       2.157
Scruz         -0.2136      0.199     -1.073      0.294      -0.624       0.197
Adjacent      -0.0765      0.017     -4.545      0.000      -0.111      -0.042
==============================================================================
Omnibus:                       11.091   Durbin-Watson:                   2.487
Prob(Omnibus):                  0.004   Jarque-Bera (JB):               11.271
Skew:                           1.002   Prob(JB):                      0.00357
Kurtosis:                       5.236   Cond. No.                         105.
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Modelo só com intercepto (modelo nulo)

In [75]:
# Equivalente a lm(Species ~ 1, galapagos) no R
model3 = smf.ols("Species ~ 1", data=galapagos).fit()
model3.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                Species   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                       nan
Date:                Thu, 18 Jun 2026   Prob (F-statistic):                nan
Time:                        22:59:03   Log-Likelihood:                -184.31
No. Observations:                  30   AIC:                             370.6
Df Residuals:                      29   BIC:                             372.0
Df Model:                           0                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     85.2333     20.929      4.072      0.000      42.429     128.038
==============================================================================
Omnibus:                       18.925   Durbin-Watson:                   1.490
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               22.278
Skew:                           1.796   Prob(JB):                     1.45e-05
Kurtosis:                       5.218   Cond. No.                         1.00
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""